## Construcción de stop times, trips y stops desde shapes

In [2]:
import pandas as pd
import geopandas as gpd

## Parámetros

In [3]:
ciudad = "tampico"

In [4]:
#path_gtfs_files = f"./data/raw_gtfs/{ciudad}/"
#path_make_gtfs_files = f"../2-generacion-archivos-make_gtfs/data/proc/{ciudad}/"

In [5]:
path_make_gtfs_files= f"../3-generacion-archivos-make_gtfs/data/proc/{ciudad}/"
path_gtfs_files = f"../4-make_gtfs/data/{ciudad}/"
                            

In [6]:
import os
path_exported_modified_files= f"./data/proc_gtfs/{ciudad}_modified_gtfs/"
os.makedirs(path_exported_modified_files, exist_ok=True)

In [7]:
dwell_time_station_minutes = 0.2 # minutes 

In [8]:
distancia_entre_estaciones = 200

In [9]:
velocidad_kmh = 28.13 # pagina 205 acb

Con las velocidades de cada ruta y las longitudes que recorren, se estimó un promedio ponderado de
velocidades, teniendo como resultado 22.91 Km/h para la SA y 28.13 Km/h para la SSP.

## Lectura de archivos

### Leer shapes

In [10]:
shapes_file = gpd.read_file(path_make_gtfs_files + "shapes.geojson", driver="GeoJSON")
shapes_file.head()

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [11]:
shapes_file.rename(columns={"data.route.shortName": "route_short_name", 
                            "data.route.longName": "route_long_name"}, inplace=True)

shapes_file.drop(columns=["route_long_name"], inplace=True)
shapes_file.head()


,route_short_name,shape_id,geometry
0,74,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [12]:
shapes_file["route_short_name"].nunique()

121

### Leer routes

In [13]:
gpd.read_file("../3-generacion-archivos-make_gtfs/data/proc/tampico/shapes.geojson")

,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."
...,...,...,...,...
116,11,Tampico - Colonias - El Fuerte - Penal por Av....,shape_11,"LINESTRING (-98.07474 22.43424, -98.07414 22.4..."
117,114,López Mateos - Margaritas - Tampico por Av. Hi...,shape_114,"LINESTRING (-98.06821 22.5432, -98.07441 22.50..."
118,111A,Tampico - Colonias - El Fuerte - Penal por Av....,shape_111A,"LINESTRING (-98.07474 22.43424, -98.07414 22.4..."
119,108A,Cuauhtémoc - Tampico - Univ. Politécnica de Al...,shape_108A,"LINESTRING (-98.15707 22.54366, -98.14982 22.5..."


In [14]:
routes_gtfs = pd.read_csv(path_gtfs_files + "routes.txt")
print(routes_gtfs["route_short_name"].nunique())
routes_gtfs.head()

121


,route_short_name,route_long_name,route_type,route_id
0,74,Isleta Pérez,2,r74
1,76,Cascajal,2,r76
2,81,Golfo,2,r81
3,89A,Altamira - Tampiquito por Soriana,2,r89A
4,67,Central Camionera - Madero,2,r67


### crear archivo de routes geoespacial

In [15]:
routes_gtfs["route_short_name"].nunique()

121

In [16]:
routes_gtfs = pd.merge(routes_gtfs,
                       shapes_file , on="route_short_name")

routes_gtfs = gpd.GeoDataFrame(routes_gtfs, geometry=routes_gtfs.geometry)
routes_gtfs.head()

,route_short_name,route_long_name,route_type,route_id,shape_id,geometry
0,74,Isleta Pérez,2,r74,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,2,r76,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,2,r81,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,2,r89A,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,2,r67,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [17]:
#routes_gtfs = routes_gtfs[routes_gtfs["route_id"]=="r36"] # r36

In [18]:
# routes_gtfs.to_file("./temp.geojson", driver="GeoJSON")

In [19]:

# Asegúrate de que está en 4326 (grados)
print(routes_gtfs.crs)  # debería decir EPSG:4326

# Reproyectar a UTM 14N (metros)
routes_m = routes_gtfs.to_crs(epsg=32614)

# Calcular longitud en metros
routes_m["length_m"] = routes_m.geometry.length

routes_m[["route_id", "route_short_name", "length_m"]].head()

EPSG:4326


,route_id,route_short_name,length_m
0,r74,74,3975.775449
1,r76,76,5008.043778
2,r81,81,7057.923626
3,r89A,89A,7208.457913
4,r67,67,7498.392531


In [20]:
# routes_gtfs.to_file("./routes_temp.geojson", driver="GeoJSON")

In [21]:
routes_m["route_id"].nunique()

121

## Generar stops y segmentos
Toma el linestring y lo divide en N segmentos

In [22]:
import math
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString, MultiPoint
from shapely.ops import linemerge, split

# --- substring de shapely si está disponible (Shapely >=1.8 / 2.x) ---
try:
    from shapely.ops import substring as _shp_substring
except Exception:
    _shp_substring = None

# ================= Utilidades =================

def _as_linestring(geom):
    """Devuelve una LineString (une MultiLineString en 1 línea si es posible)."""
    if geom.geom_type == "LineString":
        return geom
    if geom.geom_type == "MultiLineString":
        m = linemerge(geom)
        return m if m.geom_type == "LineString" else max(m.geoms, key=lambda L: L.length)
    return geom  # en casos no esperados

def _equidistant_measures(line, spacing_m):
    """Medidas a lo largo de la línea cada spacing_m, incluyendo 0 y L final."""
    L = float(line.length)
    if L == 0:
        return [0.0]
    dists = [0.0]
    k = 1
    while k * spacing_m < L - 1e-6:
        dists.append(k * spacing_m)
        k += 1
    # asegurar extremo
    if not math.isclose(dists[-1], L, abs_tol=0.05):
        dists.append(L)
    else:
        dists[-1] = L
    return dists

def _substring_by_measures(line, d0, d1):
    """Subtramo de 'line' entre medidas d0 y d1 siguiendo el trazo."""
    if _shp_substring is not None:
        return _shp_substring(line, d0, d1, normalized=False)
    # Fallback: cortar con puntos interpolados y elegir el tramo más cercano en longitud
    p0, p1 = line.interpolate(d0), line.interpolate(d1)
    parts = list(split(line, MultiPoint([p0, p1])).geoms)
    if len(parts) == 1:
        return parts[0]
    return min(parts, key=lambda s: abs(s.length - abs(d1 - d0)))

def _utm_crs_from_gdf(gdf):
    """Elige un CRS métrico UTM apropiado (para medir en metros)."""
    g4326 = gdf.to_crs(4326)
    cen = g4326.unary_union.centroid
    lon, lat = float(cen.x), float(cen.y)
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"

# ============== 1) Paradas equidistantes (con medida) ==============

def generar_paradas_equidistantes(routes_gdf, distancia_m=100,
                                  route_id_col="route_id",
                                  route_name_col="route_short_name",
                                  shape_id_col="shape_id"):
    """
    Crea puntos cada 'distancia_m' siguiendo el trazo de cada ruta,
    y guarda 'measure_m' (distancia acumulada a lo largo de la línea).
    """
    if routes_gdf.crs is None:
        raise ValueError("routes_gdf sin CRS. Define uno (p. ej. EPSG:4326).")

    crs_m = _utm_crs_from_gdf(routes_gdf)
    g_m = routes_gdf.to_crs(crs_m)

    rows = []
    use_shape = shape_id_col in g_m.columns

    for _, r in g_m.iterrows():
        rid = r[route_id_col]
        rname = r.get(route_name_col, None)
        sid = r.get(shape_id_col, None) if use_shape else None
        line = _as_linestring(r.geometry)
        if not line or line.is_empty:
            continue

        measures = _equidistant_measures(line, distancia_m)
        for i, d in enumerate(measures):
            pt = line.interpolate(d)
            stop_id = f"st_{rid}_{sid}_{i:04d}" if use_shape else f"st_{rid}_{i:04d}"
            rows.append({
                "route_id": rid,
                "route_short_name": rname,
                "shape_id": sid if use_shape else None,
                "stop_seq": i,
                "measure_m": float(d),
                "stop_id": stop_id,
                "geometry": pt
            })

    stops_m = gpd.GeoDataFrame(rows, geometry="geometry", crs=crs_m)
    return stops_m.to_crs(routes_gdf.crs)

# ============== 2) Segmentos que SIGUEN el trazo ==============

def construir_segmentos_sobre_trazo(routes_gdf, stops_gdf,
                                    route_id_col="route_id",
                                    shape_id_col="shape_id",
                                    stop_id_col="stop_id"):
    """
    Construye segmentos como SUBTRAMOS del trazado original entre medidas consecutivas.
    Si falta 'measure_m' en stops_gdf, la calcula proyectando cada punto en la línea.
    """
    if routes_gdf.crs is None or stops_gdf.crs is None:
        raise ValueError("GeoDataFrames deben tener CRS definido.")

    crs_m = _utm_crs_from_gdf(routes_gdf)
    routes_m = routes_gdf.to_crs(crs_m).copy()
    stops_m = stops_gdf.to_crs(crs_m).copy()

    use_shape = shape_id_col in routes_m.columns

    seg_rows = []
    for _, rr in routes_m.iterrows():
        rid = rr[route_id_col]
        sid = rr.get(shape_id_col, None) if use_shape else None
        line = _as_linestring(rr.geometry)
        if not line or line.is_empty:
            continue

        # filtrar paradas de esta ruta/(shape)
        mask = (stops_m[route_id_col] == rid)
        if use_shape:
            mask &= (stops_m[shape_id_col] == sid)
        sub = stops_m.loc[mask].copy()
        if sub.empty:
            continue

        # asegurar measure_m
        if "measure_m" not in sub.columns:
            sub["measure_m"] = sub.geometry.apply(lambda p: float(line.project(p)))

        sub = sub.sort_values("measure_m").reset_index(drop=True)

        # construir subtramos siguiendo el trazo
        for i in range(len(sub) - 1):
            d0 = float(sub.loc[i, "measure_m"])
            d1 = float(sub.loc[i + 1, "measure_m"])
            if not (d1 > d0 + 1e-9):
                continue  # evita segmentos nulos o invertidos
            seg_geom = _substring_by_measures(line, d0, d1)
            seg_id = f"seg_{rid}_{sid}_{i:04d}" if use_shape else f"seg_{rid}_{i:04d}"
            seg_rows.append({
                "route_id": rid,
                "shape_id": sid if use_shape else None,
                "segment_seq": i,
                "segment_id": seg_id,
                "from_stop_id": sub.loc[i, stop_id_col],
                "to_stop_id": sub.loc[i + 1, stop_id_col],
                "from_measure_m": d0,
                "to_measure_m": d1,
                "length_m": float(seg_geom.length),
                "geometry": seg_geom
            })

    segs_m = gpd.GeoDataFrame(seg_rows, geometry="geometry", crs=crs_m)
    return segs_m.to_crs(routes_gdf.crs)

In [23]:
stops = generar_paradas_equidistantes(routes_gtfs, distancia_m=distancia_entre_estaciones,
                                      route_id_col="route_id",
                                      route_name_col="route_short_name",
                                      shape_id_col="shape_id")

stops.head()

/var/folders/4n/2rt3b55x0hl9yd7kb198nxd80000gn/T/ipykernel_92000/2881612347.py:55: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cen = g4326.unary_union.centroid
/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/set_operations.py:421: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


,route_id,route_short_name,shape_id,stop_seq,measure_m,stop_id,geometry
0,r74,74,shape_74,0,0.0,st_r74_shape_74_0000,POINT (-97.84481 22.20913)
1,r74,74,shape_74,1,200.0,st_r74_shape_74_0001,POINT (-97.84429 22.21028)
2,r74,74,shape_74,2,400.0,st_r74_shape_74_0002,POINT (-97.84271 22.21133)
3,r74,74,shape_74,3,600.0,st_r74_shape_74_0003,POINT (-97.8428 22.21134)
4,r74,74,shape_74,4,800.0,st_r74_shape_74_0004,POINT (-97.84439 22.21031)


In [24]:
segments_gdf = construir_segmentos_sobre_trazo(routes_gtfs, stops,
                                           route_id_col="route_id",
                                           shape_id_col="shape_id",
                                           stop_id_col="stop_id")
segments_gdf.head()

/var/folders/4n/2rt3b55x0hl9yd7kb198nxd80000gn/T/ipykernel_92000/2881612347.py:55: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cen = g4326.unary_union.centroid
/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/set_operations.py:421: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


,route_id,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,length_m,geometry
0,r74,shape_74,0,seg_r74_shape_74_0000,st_r74_shape_74_0000,st_r74_shape_74_0001,0.0,200.0,200.0,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,r74,shape_74,1,seg_r74_shape_74_0001,st_r74_shape_74_0001,st_r74_shape_74_0002,200.0,400.0,200.0,"LINESTRING (-97.84429 22.21028, -97.84381 22.2..."
2,r74,shape_74,2,seg_r74_shape_74_0002,st_r74_shape_74_0002,st_r74_shape_74_0003,400.0,600.0,200.0,"LINESTRING (-97.84271 22.21133, -97.84196 22.2..."
3,r74,shape_74,3,seg_r74_shape_74_0003,st_r74_shape_74_0003,st_r74_shape_74_0004,600.0,800.0,200.0,"LINESTRING (-97.8428 22.21134, -97.84439 22.21..."
4,r74,shape_74,4,seg_r74_shape_74_0004,st_r74_shape_74_0004,st_r74_shape_74_0005,800.0,1000.0,200.0,"LINESTRING (-97.84439 22.21031, -97.84485 22.2..."


### Calcular tiempo en recorrer cada segmento

In [25]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import CRS

def _utm_from_centroid(gdf: gpd.GeoDataFrame) -> CRS:
    g4326 = gdf.to_crs(4326)
    c = g4326.union_all().centroid
    lon, lat = float(c.x), float(c.y)
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return CRS.from_epsg(epsg)

def _crs_is_metric(crs: CRS) -> bool:
    try:
        return any(ai.unit_name.lower().startswith(("metre", "meter")) for ai in crs.axis_info)
    except Exception:
        return False

def add_time_kmh_min(
    segments_gdf: gpd.GeoDataFrame,
    speed_kmh,                 # float | dict | pd.Series (por ruta) | pd.Series (por fila)
    by: str = "route_id",      # columna para mapear velocidad cuando sea dict/Series por ruta
    length_col: str = "length_m",
    inplace: bool = False
) -> gpd.GeoDataFrame:
    """
    Añade SOLO:
      - speed_kmh  (km/h)
      - time_min   (minutos)
    """
    if "geometry" not in segments_gdf.columns:
        raise ValueError("segments_gdf debe tener 'geometry'.")

    g = segments_gdf if inplace else segments_gdf.copy()

    # 1) Longitud en metros (si no existe, calcularla)
    if length_col in g.columns:
        len_m = g[length_col].to_numpy(dtype=float)
    else:
        if g.crs is None:
            raise ValueError("El GeoDataFrame no tiene CRS (ej. EPSG:4326).")
        crs_in = CRS.from_user_input(g.crs)
        crs_m = g.crs if _crs_is_metric(crs_in) else _utm_from_centroid(g)
        len_m = g.to_crs(crs_m).geometry.length.to_numpy()
        g[length_col] = len_m

    # 2) Resolver velocidad (km/h) vectorizada
    if np.isscalar(speed_kmh):
        v_kmh = np.full(len(g), float(speed_kmh), dtype=float)
    elif isinstance(speed_kmh, dict):
        v_kmh = pd.Series(speed_kmh).reindex(g[by]).to_numpy(dtype=float)
    elif isinstance(speed_kmh, pd.Series):
        v_kmh = (speed_kmh.to_numpy(dtype=float) if speed_kmh.index.equals(g.index)
                 else speed_kmh.reindex(g[by]).to_numpy(dtype=float))
    else:
        raise TypeError("speed_kmh debe ser float, dict o pd.Series")

    if np.any(~np.isfinite(v_kmh)) or np.any(v_kmh <= 0):
        raise ValueError("Velocidades km/h inválidas (faltantes o <= 0).")

    # 3) Tiempo en minutos (sin columnas intermedias)
    # time_min = (dist_km / kmh) * 60 = (len_m/1000) * 60 / v_kmh
    g["speed_kmh"] = v_kmh
    g["time_min_travel"]  = (g[length_col].to_numpy(dtype=float) / 1000.0) * 60.0 / v_kmh
    return g

In [26]:
# 1) Misma velocidad para todos (22 km/h)
#segments_con_tiempo = add_travel_time_kmh(segments_gdf, speed_kmh=22)

# 2) Velocidad por ruta (dict)
#vel_por_ruta = {"r74": 18, "r76": 22, "r81": 20}
#segments_con_tiempo = add_travel_time_kmh(segments_gdf, speed_kmh=vel_por_ruta)

# 3) Velocidad por fila (columna existente)
#segments_con_tiempo = add_travel_time_kmh(segments_gdf, speed_kmh=segments_gdf["v_kmh"])

In [27]:
segments_con_tiempo = add_time_kmh_min(segments_gdf, speed_kmh=velocidad_kmh)

segments_con_tiempo["dwell_time"] = dwell_time_station_minutes

segments_con_tiempo["total_time"] = segments_con_tiempo["time_min_travel"]  + segments_con_tiempo["dwell_time"] 
segments_con_tiempo.head(5)

,route_id,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,length_m,geometry,speed_kmh,time_min_travel,dwell_time,total_time
0,r74,shape_74,0,seg_r74_shape_74_0000,st_r74_shape_74_0000,st_r74_shape_74_0001,0.0,200.0,200.0,"LINESTRING (-97.84481 22.20913, -97.84525 22.2...",28.13,0.426591,0.2,0.626591
1,r74,shape_74,1,seg_r74_shape_74_0001,st_r74_shape_74_0001,st_r74_shape_74_0002,200.0,400.0,200.0,"LINESTRING (-97.84429 22.21028, -97.84381 22.2...",28.13,0.426591,0.2,0.626591
2,r74,shape_74,2,seg_r74_shape_74_0002,st_r74_shape_74_0002,st_r74_shape_74_0003,400.0,600.0,200.0,"LINESTRING (-97.84271 22.21133, -97.84196 22.2...",28.13,0.426591,0.2,0.626591
3,r74,shape_74,3,seg_r74_shape_74_0003,st_r74_shape_74_0003,st_r74_shape_74_0004,600.0,800.0,200.0,"LINESTRING (-97.8428 22.21134, -97.84439 22.21...",28.13,0.426591,0.2,0.626591
4,r74,shape_74,4,seg_r74_shape_74_0004,st_r74_shape_74_0004,st_r74_shape_74_0005,800.0,1000.0,200.0,"LINESTRING (-97.84439 22.21031, -97.84485 22.2...",28.13,0.426591,0.2,0.626591


In [28]:
#segments_con_tiempo[segments_con_tiempo["route_id"]=="r119"].to_file("./route_r119.geojson", driver="GeoJSON")

## Contruir stop times

In [29]:
import pandas as pd
import numpy as np

def build_stop_times_from_segments(
    segments_con_tiempo: pd.DataFrame,
    start_time_str: str = "00:00:00",
    trip_id_fmt: str = "{route_id}_trip_00",
    dwell_unit: str = "minutes",      # "minutes" o "seconds"
    length_col: str = "length_m",     # columna con la distancia de cada segmento (m)
    add_leg_distance: bool = True     # incluir distancia del tramo previo en cada fila
):
    req = {"route_id","segment_seq","from_stop_id","to_stop_id","time_min_travel","dwell_time", length_col}
    faltan = req - set(segments_con_tiempo.columns)
    if faltan:
        raise ValueError(f"Faltan columnas en segments_con_tiempo: {faltan}")

    seg = segments_con_tiempo.sort_values(["route_id","segment_seq"]).copy()

    def mins_to_hhmmss(mins_array):
        secs = np.rint(np.asarray(mins_array, dtype=float) * 60.0).astype(int)
        h = secs // 3600
        m = (secs % 3600) // 60
        s = secs % 60
        return pd.Series([f"{hh:02d}:{mm:02d}:{ss:02d}" for hh,mm,ss in zip(h,m,s)], index=np.arange(len(secs)))

    # inicio en minutos
    h0, m0, s0 = map(int, start_time_str.split(":"))
    t0_min = h0*60 + m0 + s0/60.0

    out_rows = []

    for rid, g in seg.groupby("route_id", sort=False):
        g = g.sort_values("segment_seq").reset_index(drop=True)
        n = len(g)
        if n == 0:
            continue

        # IDs de paradas: from del 1er segmento + todos los to
        stop_ids = pd.Index(
            np.concatenate(([g["from_stop_id"].iloc[0]], g["to_stop_id"].to_numpy(dtype=object)))
        )

        # tiempos (min)
        travel = g["time_min_travel"].to_numpy(dtype=float)     # n
        dwell  = g["dwell_time"].to_numpy(dtype=float)          # n
        if dwell_unit.lower().startswith("sec"):
            dwell = dwell / 60.0

        # distancias de cada segmento (m)
        seg_len = g[length_col].to_numpy(dtype=float)           # n
        # acumulado por parada (n+1): 0 en la primera parada, luego cumsum
        dist_cum = np.concatenate(([0.0], np.cumsum(seg_len)))  # tamaño n+1

        # cronología
        arr = np.empty(n+1, dtype=float)
        arr[0] = t0_min
        if n > 0:
            increments = dwell + travel
            arr[1:] = arr[0] + np.cumsum(increments)

        dep = arr.copy()
        if n > 0:
            dep[:n] = arr[:n] + dwell[:n]
        dep[n] = arr[n]

        stop_seq = np.arange(1, n+2, dtype=int)
        arrival_time = mins_to_hhmmss(arr)
        departure_time = mins_to_hhmmss(dep)

        data = {
            "trip_id": trip_id_fmt.format(route_id=rid),
            "stop_id": stop_ids,
            "stop_sequence": stop_seq,
            "arrival_time": arrival_time,
            "departure_time": departure_time,
            "timepoint": 1,
            # distancia recorrida acumulada hasta cada parada
            "distance_m": dist_cum
        }

        if add_leg_distance:
            # distancia del tramo previo (0 en la primera parada)
            leg_m = np.concatenate(([0.0], seg_len))
            data["leg_distance_m"] = leg_m

        df_route = pd.DataFrame(data)
        out_rows.append(df_route)

    stop_times = pd.concat(out_rows, ignore_index=True).sort_values(["trip_id","stop_sequence"])
    return stop_times


import pandas as pd
import numpy as np

def add_leg_speeds(
    stop_times: pd.DataFrame,
    time_from_col: str = "departure_time",   # tiempo en la parada i
    time_to_col: str   = "arrival_time",     # tiempo en la parada i+1
    distance_col: str  = "distance_m",       # distancia acumulada (m)
):
    """
    Agrega columnas por tramo (entre paradas consecutivas de un mismo trip_id):
      - leg_distance_m
      - leg_time_min
      - speed_mps
      - speed_kmh

    Requisitos en stop_times: ['trip_id', 'stop_sequence', time_from_col, time_to_col, distance_col]
    """

    req = {"trip_id", "stop_sequence", time_from_col, time_to_col, distance_col}
    faltan = req - set(stop_times.columns)
    if faltan:
        raise ValueError(f"Faltan columnas en stop_times: {faltan}")

    df = stop_times.copy()

    # --- parser HH:MM:SS que soporta horas > 24 ---
    def hms_to_minutes(hms: pd.Series) -> np.ndarray:
        # Espera strings tipo 'HH:MM:SS' con HH entero (posible >24)
        h, m, s = (
            hms.str.split(":", expand=True)[0].astype(int),
            hms.str.split(":", expand=True)[1].astype(int),
            hms.str.split(":", expand=True)[2].astype(int),
        )
        return (h * 60 + m + s / 60.0).to_numpy(dtype=float)

    # Orden estable por viaje
    df = df.sort_values(["trip_id", "stop_sequence"], kind="mergesort")

    # Vectorización por grupo
    def _per_trip(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("stop_sequence").copy()

        # Distancia acumulada -> distancia de tramo (diferen cia hacia adelante)
        dist_cum = g[distance_col].to_numpy(dtype=float)
        leg_dist = np.concatenate(([np.nan], np.diff(dist_cum)))

        # Tiempos: from en i, to en i+1
        t_from_min = hms_to_minutes(g[time_from_col])
        t_to_min   = hms_to_minutes(g[time_to_col])
        # alineamos to(i+1)
        leg_time_min = np.concatenate(([np.nan], t_to_min[1:] - t_from_min[:-1]))

        # Velocidades
        with np.errstate(divide="ignore", invalid="ignore"):
            speed_mps = leg_dist / (leg_time_min * 60.0)
            speed_kmh = speed_mps * 3.6

        g["leg_distance_m"] = leg_dist
        g["leg_time_min"]   = leg_time_min
        g["speed_kmh"]      = speed_kmh

        # saneamiento: valores imposibles/negativos -> NaN
        g.loc[(g["leg_distance_m"] <= 0) | (g["leg_time_min"] <= 0), ["speed_mps","speed_kmh"]] = np.nan
        return g

    out = df.groupby("trip_id", group_keys=False, sort=False).apply(_per_trip)
    return out

In [30]:
# segments_con_tiempo = add_time_kmh_min(segments_gdf, speed_kmh=22)
# segments_con_tiempo["dwell_time"] = dwell_time_station_minutes
# segments_con_tiempo["total_time"] = segments_con_tiempo["time_min_travel"] + segments_con_tiempo["dwell_time"]

stop_times = build_stop_times_from_segments(segments_con_tiempo,
                                            start_time_str="00:00:00",
                                            trip_id_fmt="{route_id}_trip_00",     length_col="length_m",
                                                add_leg_distance=False)

stop_times = add_leg_speeds(
    stop_times,
    time_from_col="departure_time",
    time_to_col="arrival_time",
    distance_col="distance_m",
)



stop_times = stop_times[["trip_id","timepoint","stop_id","stop_sequence","arrival_time","departure_time"
]]

stop_times.head(30)

/var/folders/4n/2rt3b55x0hl9yd7kb198nxd80000gn/T/ipykernel_92000/742231488.py:159: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby("trip_id", group_keys=False, sort=False).apply(_per_trip)


,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
682,r100A_trip_00,1,st_r100A_shape_100A_0000,1,00:00:00,00:00:12
683,r100A_trip_00,1,st_r100A_shape_100A_0001,2,00:00:38,00:00:50
684,r100A_trip_00,1,st_r100A_shape_100A_0002,3,00:01:15,00:01:27
685,r100A_trip_00,1,st_r100A_shape_100A_0003,4,00:01:53,00:02:05
686,r100A_trip_00,1,st_r100A_shape_100A_0004,5,00:02:30,00:02:42
687,r100A_trip_00,1,st_r100A_shape_100A_0005,6,00:03:08,00:03:20
688,r100A_trip_00,1,st_r100A_shape_100A_0006,7,00:03:46,00:03:58
689,r100A_trip_00,1,st_r100A_shape_100A_0007,8,00:04:23,00:04:35
690,r100A_trip_00,1,st_r100A_shape_100A_0008,9,00:05:01,00:05:13
691,r100A_trip_00,1,st_r100A_shape_100A_0009,10,00:05:38,00:05:50


In [31]:
segments_con_tiempo.head()

,route_id,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,length_m,geometry,speed_kmh,time_min_travel,dwell_time,total_time
0,r74,shape_74,0,seg_r74_shape_74_0000,st_r74_shape_74_0000,st_r74_shape_74_0001,0.0,200.0,200.0,"LINESTRING (-97.84481 22.20913, -97.84525 22.2...",28.13,0.426591,0.2,0.626591
1,r74,shape_74,1,seg_r74_shape_74_0001,st_r74_shape_74_0001,st_r74_shape_74_0002,200.0,400.0,200.0,"LINESTRING (-97.84429 22.21028, -97.84381 22.2...",28.13,0.426591,0.2,0.626591
2,r74,shape_74,2,seg_r74_shape_74_0002,st_r74_shape_74_0002,st_r74_shape_74_0003,400.0,600.0,200.0,"LINESTRING (-97.84271 22.21133, -97.84196 22.2...",28.13,0.426591,0.2,0.626591
3,r74,shape_74,3,seg_r74_shape_74_0003,st_r74_shape_74_0003,st_r74_shape_74_0004,600.0,800.0,200.0,"LINESTRING (-97.8428 22.21134, -97.84439 22.21...",28.13,0.426591,0.2,0.626591
4,r74,shape_74,4,seg_r74_shape_74_0004,st_r74_shape_74_0004,st_r74_shape_74_0005,800.0,1000.0,200.0,"LINESTRING (-97.84439 22.21031, -97.84485 22.2...",28.13,0.426591,0.2,0.626591


## Archivo Frequencies

In [32]:
import pandas as pd
import numpy as np

def build_frequencies(stop_times: pd.DataFrame,
                      intervals: list,
                      save_path: str | None = None) -> pd.DataFrame:
    """
    Genera frequencies.txt a partir de stop_times.

    Parameters
    ----------
    stop_times : DataFrame con columna 'trip_id' (única por viaje base).
    intervals : lista de dicts, cada uno con:
        {
          "start_time": "HH:MM:SS",
          "end_time":   "HH:MM:SS",
          "headway_secs": 600,
          "exact_times": 0  # opcional, default=0
        }
      Puedes pasar uno o varios intervalos.
    save_path : ruta opcional para guardar CSV (e.g. "./gtfs/frequencies.txt")

    Returns
    -------
    DataFrame con columnas:
      ['trip_id','start_time','end_time','headway_secs','exact_times']
    """
    if "trip_id" not in stop_times.columns:
        raise ValueError("stop_times debe incluir la columna 'trip_id'.")

    # Trips únicos
    trips = (stop_times[["trip_id"]]
             .drop_duplicates()
             .reset_index(drop=True))

    # Normalizar/validar intervalos a DataFrame
    intervals_df = pd.DataFrame(intervals).copy()
    if intervals_df.empty:
        raise ValueError("Debes proporcionar al menos un intervalo.")
    if "headway_secs" not in intervals_df.columns:
        raise ValueError("Cada intervalo requiere 'headway_secs'.")
    # Campos por defecto
    intervals_df["exact_times"] = intervals_df.get("exact_times", 0).fillna(0).astype(int)

    # Validaciones básicas
    def _valid_hhmmss(s):
        try:
            h,m,sec = map(int, str(s).split(":"))
            return (h >= 0 and 0 <= m < 60 and 0 <= sec < 60)
        except Exception:
            return False

    bad_start = intervals_df["start_time"].apply(lambda x: not _valid_hhmmss(x))
    bad_end   = intervals_df["end_time"].apply(lambda x: not _valid_hhmmss(x))
    if bad_start.any() or bad_end.any():
        raise ValueError("Formato de hora inválido en start_time/end_time (usa HH:MM:SS; se permiten horas >=24).")

    if (intervals_df["headway_secs"] <= 0).any():
        raise ValueError("'headway_secs' debe ser > 0.")

    # Producto cartesiano trips x intervals
    trips["__key"] = 1
    intervals_df["__key"] = 1
    frequencies = (trips.merge(intervals_df, on="__key", how="inner")
                        .drop(columns="__key"))

    # Orden y tipos
    frequencies = frequencies[["trip_id", "start_time", "end_time", "headway_secs", "exact_times"]]
    frequencies["headway_secs"] = frequencies["headway_secs"].astype(int)
    frequencies["exact_times"]  = frequencies["exact_times"].astype(int)


    return frequencies

In [33]:
# Ya tienes stop_times construido con build_stop_times_from_segments(...)
stop_times = stop_times[["trip_id","timepoint","stop_id","stop_sequence","arrival_time","departure_time"]]

# Intervalo único: cada 15 min entre 06:00 y 09:00, exact_times=0
intervalos = [{
    "start_time": "06:00:00",
    "end_time":   "07:00:00",
    "headway_secs": 787, #" según intervalos"
    "exact_times": 1,
}]

frequencies = build_frequencies(stop_times, intervals=intervalos,
                                save_path="./gtfs/frequencies.txt")

frequencies = frequencies[["end_time","exact_times","headway_secs","start_time","trip_id"]]
frequencies.head()

,end_time,exact_times,headway_secs,start_time,trip_id
0,07:00:00,1,787,06:00:00,r100A_trip_00
1,07:00:00,1,787,06:00:00,r100_trip_00
2,07:00:00,1,787,06:00:00,r101A_trip_00
3,07:00:00,1,787,06:00:00,r101_trip_00
4,07:00:00,1,787,06:00:00,r103A_trip_00


## Archivo Trips

In [34]:
# leemos el archivo stops del GTFS solo para ver el formato
trips_gtfs = pd.read_csv(path_gtfs_files + "trips.txt")
print(trips_gtfs.columns)
trips_gtfs.head()

Index(['route_id', 'trip_id', 'direction_id', 'shape_id', 'service_id'], dtype='object')


,route_id,trip_id,direction_id,shape_id,service_id
0,r74,t-r74-weekday-06:00:00-1-0,1,shape_74-1,srv1111111
1,r74,t-r74-weekday-06:00:00-1-1,1,shape_74-1,srv1111111
2,r74,t-r74-weekday-06:00:00-1-2,1,shape_74-1,srv1111111
3,r74,t-r74-weekday-06:00:00-1-3,1,shape_74-1,srv1111111
4,r74,t-r74-weekday-06:00:00-1-4,1,shape_74-1,srv1111111


Limpiar archivo frequencies
Luego se usa para hacer merfge con trips generados

In [35]:
frequencies_trips = frequencies[["trip_id"]].copy()
frequencies_trips["route_id"] = frequencies_trips["trip_id"].str.split("_").str[0]

frequencies_trips.head(10)

,trip_id,route_id
0,r100A_trip_00,r100A
1,r100_trip_00,r100
2,r101A_trip_00,r101A
3,r101_trip_00,r101
4,r103A_trip_00,r103A
5,r103B_trip_00,r103B
6,r103_trip_00,r103
7,r104_trip_00,r104
8,r105A_trip_00,r105A
9,r105_trip_00,r105


In [36]:
trips_gtfs = trips_gtfs.drop_duplicates("route_id")
trips_gtfs = trips_gtfs[['route_id',  'direction_id', 'shape_id', 'service_id']]
trips_gtfs.tail()

,route_id,direction_id,shape_id,service_id
2900,r11,1,shape_11-1,srv1111111
2925,r114,1,shape_114-1,srv1111111
2950,r111A,1,shape_111A-1,srv1111111
2975,r108A,1,shape_108A-1,srv1111111
3000,r108,1,shape_108-1,srv1111111


In [37]:
trips_gtfs[trips_gtfs["route_id"]=="r119"]

,route_id,direction_id,shape_id,service_id
1750,r119,1,shape_119-1,srv1111111


In [38]:
# trips_gtfs.to_csv("./temp_trips.csv")

In [39]:
frequencies_trips.tail()

,trip_id,route_id
116,r89_trip_00,r89
117,r8_trip_00,r8
118,r90_trip_00,r90
119,r9_trip_00,r9
120,rFracc._trip_00,rFracc.


In [40]:
frequencies_trips["trip_id"].nunique()

121

In [41]:
trips_export = pd.merge(trips_gtfs, frequencies_trips, on="route_id", how="right")
#trips_export = trips_export[["route_id","service_id","trip_id","direction_id","shape_id"]]
trips_export.head()

,route_id,direction_id,shape_id,service_id,trip_id
0,r100A,1,shape_100A-1,srv1111111,r100A_trip_00
1,r100,1,shape_100-1,srv1111111,r100_trip_00
2,r101A,1,shape_101A-1,srv1111111,r101A_trip_00
3,r101,1,shape_101-1,srv1111111,r101_trip_00
4,r103A,1,shape_103A-1,srv1111111,r103A_trip_00


## Convertir a formato GTFS

### Stops
Convertimos el stops generado al formato gtfs

In [42]:
# leemos el archivo stops del GTFS solo para ver el formato
stops_gtfs = pd.read_csv(path_gtfs_files + "stops.txt")
print(stops_gtfs.columns)
stops_gtfs.head()

Index(['stop_code', 'stop_lat', 'stop_lon', 'stop_id', 'stop_name', 'zone_id',
       'parent_station', 'stop_desc', 'location_type'],
      dtype='object')


,stop_code,stop_lat,stop_lon,stop_id,stop_name,zone_id,parent_station,stop_desc,location_type
0,0,22.209196,-97.844766,stop_left_0001,Stop 1,merged_cluster,NaN,NaN,0
1,3,22.210877,-97.846005,stop_left_0004,Stop 4,merged_cluster,NaN,NaN,0
2,4,22.213285,-97.849042,stop_left_0005,Stop 5,merged_cluster,NaN,NaN,0
3,5,22.212175,-97.849720,stop_left_0006,Stop 6,merged_cluster,NaN,NaN,0
4,6,22.214650,-97.851593,stop_left_0007,Stop 7,merged_cluster,NaN,NaN,0


In [43]:
# generamos columnas y limpiamos archivos
stops.head()

,route_id,route_short_name,shape_id,stop_seq,measure_m,stop_id,geometry
0,r74,74,shape_74,0,0.0,st_r74_shape_74_0000,POINT (-97.84481 22.20913)
1,r74,74,shape_74,1,200.0,st_r74_shape_74_0001,POINT (-97.84429 22.21028)
2,r74,74,shape_74,2,400.0,st_r74_shape_74_0002,POINT (-97.84271 22.21133)
3,r74,74,shape_74,3,600.0,st_r74_shape_74_0003,POINT (-97.8428 22.21134)
4,r74,74,shape_74,4,800.0,st_r74_shape_74_0004,POINT (-97.84439 22.21031)


In [49]:
#stops[stops["route_id"]=="r7"].to_file("./stops_r7.geojson", driver="GeoJSON")

In [50]:
stops["stop_name"] = stops["stop_id"]

stops["stop_lon"] = stops["geometry"].x
stops["stop_lat"] = stops["geometry"].y

#stops.drop(columns=["route_id",  "shape_id",   "geometry"], inplace=True)
stops = stops[['stop_id', 'stop_name', 'stop_lon', 'stop_lat']]
stops.head()

,stop_id,stop_name,stop_lon,stop_lat
0,st_r74_shape_74_0000,st_r74_shape_74_0000,-97.844810,22.209130
1,st_r74_shape_74_0001,st_r74_shape_74_0001,-97.844291,22.210283
2,st_r74_shape_74_0002,st_r74_shape_74_0002,-97.842709,22.211326
3,st_r74_shape_74_0003,st_r74_shape_74_0003,-97.842802,22.211341
4,st_r74_shape_74_0004,st_r74_shape_74_0004,-97.844393,22.210307


## Exportar

In [51]:
stops.head()

,stop_id,stop_name,stop_lon,stop_lat
0,st_r74_shape_74_0000,st_r74_shape_74_0000,-97.844810,22.209130
1,st_r74_shape_74_0001,st_r74_shape_74_0001,-97.844291,22.210283
2,st_r74_shape_74_0002,st_r74_shape_74_0002,-97.842709,22.211326
3,st_r74_shape_74_0003,st_r74_shape_74_0003,-97.842802,22.211341
4,st_r74_shape_74_0004,st_r74_shape_74_0004,-97.844393,22.210307


In [52]:
frequencies.head()

,end_time,exact_times,headway_secs,start_time,trip_id
0,07:00:00,1,787,06:00:00,r100A_trip_00
1,07:00:00,1,787,06:00:00,r100_trip_00
2,07:00:00,1,787,06:00:00,r101A_trip_00
3,07:00:00,1,787,06:00:00,r101_trip_00
4,07:00:00,1,787,06:00:00,r103A_trip_00


In [53]:
trips_export.head()

,route_id,direction_id,shape_id,service_id,trip_id
0,r100A,1,shape_100A-1,srv1111111,r100A_trip_00
1,r100,1,shape_100-1,srv1111111,r100_trip_00
2,r101A,1,shape_101A-1,srv1111111,r101A_trip_00
3,r101,1,shape_101-1,srv1111111,r101_trip_00
4,r103A,1,shape_103A-1,srv1111111,r103A_trip_00


In [54]:
stops.to_csv(f"{path_exported_modified_files}/stops.txt",
             index=False)

stop_times.to_csv(f"{path_exported_modified_files}/stop_times.txt",
                        index=False)

frequencies.to_csv(f"{path_exported_modified_files}/frequencies.txt",
                        index=False)

trips_export.to_csv(f"{path_exported_modified_files}/trips.txt",
                        index=False)

In [55]:
path_exported_modified_files

'./data/proc_gtfs/tampico_modified_gtfs/'